Retreive 20 records including the last 4 columns from the GCS's file generated from ex01.ipynb
Using "gemini-2.5-flash", ask it to summarize the reports.

- Pre-requisite : Generate API keys through [Google AI Studio](https://aistudio.google.com/app/apikey)

In [1]:
import json
import os 

from dotenv import load_dotenv
from google import genai
from google.oauth2 import service_account
from google.cloud import storage

In [2]:
load_dotenv(dotenv_path='/Users/lokeshmuvva/Documents/msds692_data_acquisition_2025/Day3/.env')

True

In [3]:
genai_api_key = os.getenv("GCP_GENAI_API_KEY")
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
file_name = (f"sf_police_report/2025-09-08.json")

# print(genai_api_key)

In [4]:
def retrieve_data_from_gcs(service_account_key: str,
                           project_id: str,
                           bucket_name: str,
                           file_name: str,
                           key_list: list
                           ) -> list:
    credentials = service_account.Credentials.from_service_account_file(service_account_key)
    client = storage.Client(project=project_id,
                            credentials=credentials)
    bucket = client.bucket(bucket_name)
    file = bucket.blob(file_name)
    content = json.loads(file.download_as_string())

    output = []
    for data in content:
        row = []
        
        for key in key_list:
            row.append(data.get(key, None))
        output.append(row)
    return output

In [5]:
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
data = retrieve_data_from_gcs(service_account_key, 
                              project_id,
                              bucket_name,
                              file_name,
                              key_list)

In [6]:
filtered_data = [row[-4:] for row in data][:20]

In [7]:
client = genai.Client(api_key=genai_api_key)

In [8]:
model_name = "gemini-2.5-flash"

In [9]:
prompt_content = (f"Summarize police report from {filtered_data}")

In [10]:
response = client.models.generate_content(
    model=model_name,
    contents=prompt_content
)

In [11]:
response.text

'This police report summary covers 20 incidents across various categories and San Francisco districts.\n\n**Incident Types:**\n*   **Property Crimes (6 incidents):** This is the most frequent category, including two instances of "Theft, From Unlocked Vehicle, >$950," "Burglary, Residence, Unlawful Entry," "Theft, Other Property, $50-$200," "Vehicle, Stolen, Auto," and "Vehicle, Stolen, Motorcycle."\n*   **Violent Crimes (5 incidents):** Comprises "Battery," "Robbery, Street or Public Place, W/ Force," "Assault, Aggravated, W/ Force," "Battery with Serious Injuries," and "Robbery, W/ Force."\n*   **Miscellaneous & Other (7 incidents):** This broad category includes "False Personation," "Suspicious Occurrence," "Miscellaneous Investigation," "Violation of Restraining Order," "Municipal Police Code Violation," and "Death Report, Cause Unknown." One "Vehicle, Recovered, Auto" is also noted.\n*   **Drug-Related (2 incidents):** "Loitering Where Narcotics are Sold/Used" and "Narcotics Paraph